# VoiceDiary AI — Live Cloud GPU Platform
### Bilingual Classroom Lecture Note-Taking & Speaker Diarization Engine
**VoiceDiary © 2026 Abdul Sarim Khan. All Rights Reserved.**

High-Performance Cloud GPU inference (`large-v3-turbo` default + 6 Model Hub + `ECAPA-TDNN` + `Gemini 2.5 Flash`).

---
### Quick Start:
1. Click **Runtime → Run all** (or `Ctrl + F9`).
2. The VoiceDiary interface will render below. Record or upload audio — transcription starts automatically!

In [ ]:
# 1. Install GPU Acceleration Libraries
!pip install -q --no-cache-dir faster-whisper speechbrain gradio soundfile torchaudio


In [ ]:
# 2. Launch VoiceDiary — Full Desktop UI Parity + GPU Engine
import os, time, tempfile, json, re, urllib.request, gc
import numpy as np
import soundfile as sf
import gradio as gr
import torch
import torchaudio
from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier

# ─── Hardware ───
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (AVX2)"
device_type = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = "float16" if device_type == "cuda" else "int8"
print(f"⚡ {gpu_name} | {compute_dtype}")

# ─── Model Hub (lazy-cached, large-v3-turbo pre-warmed) ───
_model_cache = {}
def get_model(name):
    if name not in _model_cache:
        print(f"  Loading {name}…")
        os.makedirs("/content/models/whisper", exist_ok=True)
        _model_cache[name] = WhisperModel(name, device=device_type,
            compute_type=compute_dtype, num_workers=2,
            download_root="/content/models/whisper")
    return _model_cache[name]

print("Pre-warming Large-v3-Turbo …")
get_model("large-v3-turbo")
print("✓ Large-v3-Turbo ready.")

# ─── ECAPA-TDNN ───
print("Pre-warming ECAPA-TDNN …")
os.makedirs("/content/models/ecapa", exist_ok=True)
embedder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/models/ecapa",
    run_opts={"device": device_type})
print("✓ ECAPA-TDNN ready.")

# ─── Roman Urdu dict ───
_roman = {
    'آپ':'aap','کیسے':'kaisay','ہیں':'hain','کیا':'kya','کر':'kar',
    'رہے':'rahay','ہو':'ho','میں':'main','ہوں':'hoon','یہ':'yeh',
    'وہ':'woh','نہیں':'nahi','ٹھیک':'theek','شکریہ':'shukriya',
    'سلام':'salam','بہت':'bohot','اچھا':'acha','سوال':'sawaal','جواب':'jawab',
}
def to_roman(t):
    return " ".join(_roman.get(re.sub(r'[\u064B-\u065F\u0670]','',w), w) for w in t.split())

# ─── Audio loader (torchaudio GPU-accelerated) ───
def load_16k(path):
    try:
        wav, sr = torchaudio.load(path)
        if wav.shape[0]>1: wav = wav.mean(0, keepdim=True)
        if sr!=16000: wav = torchaudio.transforms.Resample(sr,16000)(wav)
        return wav.squeeze().numpy().astype(np.float32)
    except Exception:
        d,sr = sf.read(path)
        if d.ndim>1: d=d.mean(1)
        d = d.astype(np.float32)
        if sr!=16000:
            n=int(len(d)*16000/sr)
            d=np.interp(np.linspace(0,len(d),n,endpoint=False),np.arange(len(d)),d).astype(np.float32)
        return d

# ─── Core pipeline ───
COLORS = ['#6366F1','#10B981','#F59E0B','#EC4899','#06B6D4','#8B5CF6','#F97316']
MODEL_MAP = {
    'Large-v3-Turbo (809M)': 'large-v3-turbo',
    'Whisper Base (74M)': 'base',
    'Whisper Tiny (39M)': 'tiny',
    'Whisper Small (244M)': 'small',
    'Whisper Medium (769M)': 'medium',
    'Distil-Whisper (756M)': 'distil-large-v3',
}
LANG_MAP = {
    'Bilingual (Urdu + English)': (None, False),
    'Pure Urdu Script (اردو)': ('ur', False),
    'English Only': ('en', False),
    'Roman Urdu (Latin)': ('ur', True),
}

def transcribe(audio_path, model_choice, lang_choice, thresh_pct, vad_ms):
    if not audio_path or not os.path.exists(audio_path):
        empty = "<div class='vd-empty'><svg width='40' height='40' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.4'><path d='M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z'/></svg><p>Lecture transcript will stream here</p><span>Record audio or upload a file to begin</span></div>"
        sidebar = "<div class='vd-empty-sm'><svg width='28' height='28' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.3'><path d='M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z'/><path d='M19 10v2a7 7 0 0 1-14 0v-2'/></svg><p>No speakers detected</p><span>Start recording to identify speakers</span></div>"
        return empty, sidebar, "", None, None, None, None

    t0 = time.time()
    data = load_16k(audio_path)
    dur = len(data)/16000.0
    mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
    model = get_model(mkey)
    target_lang, is_roman = LANG_MAP.get(lang_choice, (None, False))
    thresh = float(thresh_pct)/100.0

    segs, info = model.transcribe(data,
        beam_size=1, best_of=1, temperature=0.0,
        language=target_lang, without_timestamps=False,
        vad_filter=True, vad_parameters=dict(min_silence_duration_ms=int(vad_ms)))

    profiles = {}; nxt=1
    html_parts=[]; plain=[]; srt_parts=[]; json_arr=[]; si=1

    for seg in segs:
        txt = seg.text.strip()
        if not txt: continue
        if is_roman: txt = to_roman(txt)
        s0,s1 = seg.start, seg.end
        chunk = data[int(s0*16000):int(s1*16000)]
        spk = 1
        if len(chunk)>=8000:
            try:
                with torch.inference_mode():
                    w = torch.from_numpy(chunk).float().unsqueeze(0).to(device_type)
                    e = embedder.encode_batch(w).squeeze().detach().cpu().numpy()
                    en = e/(np.linalg.norm(e) or 1.)
                bid,bsim = None,-1.
                for sid,embs in profiles.items():
                    ms_ = max(float(np.dot(en,x)) for x in embs) if embs else 0
                    if ms_>bsim: bsim,bid=ms_,sid
                if bid and bsim>=thresh:
                    spk=bid
                    if len(profiles[spk])<50: profiles[spk].append(en)
                else:
                    spk=nxt; profiles[spk]=[en]; nxt+=1
            except: pass

        c = COLORS[(spk-1)%len(COLORS)]
        ts = f"{int(s0//60):02d}:{int(s0%60):02d}"
        urdu = any('\u0600'<=ch<='\u06FF' for ch in txt)
        rtl = " vd-rtl" if urdu else ""

        html_parts.append(f"""<div class="vd-node" style="border-left-color:{c}">
<div class="vd-node-head"><span class="vd-dot" style="background:{c}"></span><strong style="color:{c}">Speaker {spk}</strong><span class="vd-ts">[{ts}]</span></div>
<div class="vd-node-text{rtl}">{txt}</div></div>""")
        plain.append(f"[{ts}] Speaker {spk}: {txt}")

        def srt_t(s):
            return f"{int(s//3600):02d}:{int(s%3600//60):02d}:{int(s%60):02d},{int((s-int(s))*1000):03d}"
        srt_parts.append(f"{si}\n{srt_t(s0)} --> {srt_t(s1)}\n[Speaker {spk}]: {txt}\n")
        json_arr.append({"speaker":f"Speaker {spk}","id":spk,"start":round(s0,2),"end":round(s1,2),"time":ts,"text":txt})
        si+=1

    elapsed = time.time()-t0

    # Sidebar
    sb = []
    for sid,embs in profiles.items():
        cc = COLORS[(sid-1)%len(COLORS)]
        sb.append(f"""<div class="vd-spk-card"><div class="vd-spk-av" style="background:{cc}">S{sid}</div><div><div class="vd-spk-name">Speaker {sid}</div><div class="vd-spk-meta">{len(embs)} voiceprint{'s' if len(embs)>1 else ''}</div></div></div>""")
    if not sb:
        sb.append("<div class='vd-empty-sm'><p>No speakers detected</p></div>")

    # Stats footer
    stats = f"""<div class="vd-stats"><span>Engine: {mkey} · {gpu_name}</span><span>{dur:.1f}s → {elapsed:.1f}s ({dur/max(.01,elapsed):.1f}× real-time)</span></div>"""

    transcript_html = "\n".join(html_parts) + stats
    sidebar_html = "\n".join(sb)

    # Build export files
    files = {}
    for ext, content in [
        ('.md', f"# VoiceDiary Lecture Notes\n\n"+"\n\n".join(plain)),
        ('.txt', "\n".join(plain)),
        ('.srt', "\n".join(srt_parts)),
        ('.json', json.dumps(json_arr, indent=2, ensure_ascii=False))
    ]:
        f = tempfile.NamedTemporaryFile(mode='w', suffix=ext, delete=False, encoding='utf-8', prefix='VoiceDiary_')
        f.write(content); f.close()
        files[ext] = f.name

    # Export section HTML with clean download links
    export_html = ""
    if plain:
        export_html = f"""<div class="vd-export-grid">
<a href="/file={files['.md']}" download class="vd-export-btn"><svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"/><polyline points="7 10 12 15 17 10"/><line x1="12" y1="15" x2="12" y2="3"/></svg>Markdown (.md)</a>
<a href="/file={files['.txt']}" download class="vd-export-btn"><svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/></svg>Plain Text (.txt)</a>
<a href="/file={files['.srt']}" download class="vd-export-btn"><svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="2" y="2" width="20" height="20" rx="2.18" ry="2.18"/><line x1="7" y1="2" x2="7" y2="22"/><line x1="17" y1="2" x2="17" y2="22"/><line x1="2" y1="12" x2="22" y2="12"/></svg>Subtitles (.srt)</a>
<a href="/file={files['.json']}" download class="vd-export-btn"><svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><polyline points="16 18 22 12 16 6"/><polyline points="8 6 2 12 8 18"/></svg>JSON Data (.json)</a>
</div>"""

    return transcript_html, sidebar_html, export_html, "\n".join(plain), files.get('.md'), files.get('.txt'), files.get('.srt')

def gemini_summary(text, key):
    if not text or not text.strip(): return "*No transcript yet.*"
    api_key = (key or "").strip() or os.environ.get("GEMINI_API_KEY","")
    if not api_key: return "*Paste your Gemini API key in the sidebar to generate study notes.*"
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
    payload = {"contents":[{"parts":[{"text":f"You are VoiceDiary AI, an academic study summarizer. Analyze this diarized lecture transcript and produce:\n1. Executive Overview\n2. Core Concepts\n3. Key Exam Points\n4. Q&A Highlights\n\nTranscript:\n{text}"}]}]}
    try:
        req = urllib.request.Request(url, data=json.dumps(payload).encode(), headers={"Content-Type":"application/json"})
        with urllib.request.urlopen(req, timeout=30) as r:
            return json.loads(r.read())["candidates"][0]["content"]["parts"][0]["text"]
    except Exception as e: return f"*Error: {e}*"

# ─── CSS: Exact Desktop Design System ───
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Fira+Code:wght@400;500;600&family=Noto+Nastaliq+Urdu:wght@400;700&display=swap');

/* === Root Design Tokens (from desktop styles.css) === */
:root {
  --bg-base: #080C14; --bg-surface: #0F172A; --bg-elevated: #1E293B;
  --bg-card: rgba(15,23,42,.75); --bg-glass: rgba(15,23,42,.65);
  --border: rgba(255,255,255,.08); --border-md: rgba(255,255,255,.14);
  --text-1: #F8FAFC; --text-2: #94A3B8; --text-3: #64748B;
  --accent: #6366F1; --accent-hover: #4F46E5;
  --accent-grad: linear-gradient(135deg,#6366F1,#8B5CF6);
  --glow: rgba(99,102,241,.25); --red: #EF4444; --green: #10B981; --amber: #F59E0B;
  --r-sm:6px; --r-md:10px; --r-lg:14px; --r-xl:20px; --r-full:9999px;
}

/* === Global === */
body, .gradio-container {
  background: var(--bg-base) !important;
  background-image: radial-gradient(ellipse 80% 50% at 50% -20%,rgba(99,102,241,.15),transparent),
                     radial-gradient(ellipse 60% 40% at 90% 90%,rgba(139,92,246,.08),transparent) !important;
  font-family: 'Plus Jakarta Sans',-apple-system,sans-serif !important;
  color: var(--text-1) !important;
  max-width: 1440px !important; margin: 0 auto !important;
}

/* === Override ALL Gradio native chrome === */
.gradio-container .block { background: transparent !important; border: none !important; box-shadow: none !important; }
.gr-group { background: var(--bg-card) !important; border: 1px solid var(--border) !important; border-radius: var(--r-xl) !important; padding: 18px !important; backdrop-filter: blur(16px); }
.gr-panel, .gr-box, .gr-form { background: transparent !important; border: none !important; }
.gr-input, .gr-text-input, textarea, select, .gr-dropdown { background: var(--bg-elevated) !important; border: 1px solid var(--border) !important; border-radius: var(--r-md) !important; color: var(--text-1) !important; font-family: inherit !important; }
.gr-button { border-radius: var(--r-md) !important; font-family: inherit !important; font-weight: 600 !important; }
label, .gr-label { color: var(--text-2) !important; font-weight: 600 !important; font-size: 12px !important; letter-spacing: .03em !important; }
.tabs { border: none !important; }
.tab-nav { background: transparent !important; border: none !important; }
.tab-nav button { background: transparent !important; border: 1px solid transparent !important; border-bottom: none !important; border-radius: var(--r-lg) var(--r-lg) 0 0 !important; color: var(--text-2) !important; font-weight: 600 !important; font-size: 13px !important; padding: 10px 18px !important; }
.tab-nav button.selected { background: var(--bg-card) !important; border-color: var(--border) !important; color: var(--text-1) !important; }
.tabitem { background: transparent !important; border: none !important; }

/* Slider */
input[type=range] { accent-color: var(--accent) !important; }
.gr-slider input[type=number] { background: var(--bg-elevated) !important; border: 1px solid var(--border) !important; color: var(--text-1) !important; border-radius: var(--r-sm) !important; width: 52px !important; }

/* Hide Gradio file component chrome completely */
.gr-file { display: none !important; }

/* Footer hide */
footer { display: none !important; }

/* === VoiceDiary Custom Components === */

/* Header */
.vd-header {
  height: 60px; background: var(--bg-glass); backdrop-filter: blur(20px);
  border: 1px solid var(--border); border-radius: var(--r-xl);
  display: flex; align-items: center; justify-content: space-between;
  padding: 0 20px; margin-bottom: 14px;
}
.vd-header-left { display: flex; align-items: center; gap: 12px; }
.vd-logo {
  width: 36px; height: 36px; background: var(--accent-grad); border-radius: var(--r-md);
  display: flex; align-items: center; justify-content: center; color: #fff;
  box-shadow: 0 0 14px var(--glow);
}
.vd-title { font-size: 17px; font-weight: 800; color: #fff; letter-spacing: -.02em; margin: 0; }
.vd-subtitle { font-size: 11px; color: var(--text-2); font-weight: 500; }
.vd-pill {
  display: inline-flex; align-items: center; gap: 6px;
  background: rgba(16,185,129,.1); border: 1px solid rgba(16,185,129,.25);
  padding: 5px 14px; border-radius: var(--r-full);
  font-size: 11px; font-weight: 700; color: var(--green);
  font-family: 'Fira Code',monospace;
}
.vd-pill-dot { width: 6px; height: 6px; border-radius: 50%; background: var(--green); box-shadow: 0 0 6px rgba(16,185,129,.6); }

/* Section headers */
.vd-section-hdr {
  font-size: 11px; font-weight: 800; color: var(--text-2);
  letter-spacing: .06em; margin-bottom: 14px;
  display: flex; justify-content: space-between; align-items: center;
}
.vd-badge { background: rgba(99,102,241,.15); color: #818CF8; font-size: 11px; font-weight: 700; padding: 2px 8px; border-radius: var(--r-full); }

/* Speaker cards */
.vd-spk-card {
  display: flex; align-items: center; gap: 12px;
  padding: 10px 14px; border-radius: var(--r-lg);
  background: rgba(255,255,255,.02); border: 1px solid rgba(255,255,255,.06);
  margin-bottom: 8px; transition: .15s ease;
}
.vd-spk-card:hover { background: rgba(255,255,255,.05); border-color: var(--border-md); }
.vd-spk-av {
  width: 34px; height: 34px; border-radius: 50%;
  display: flex; align-items: center; justify-content: center;
  font-weight: 800; font-size: 13px; color: #fff; flex-shrink: 0;
}
.vd-spk-name { font-size: 13px; font-weight: 600; color: var(--text-1); }
.vd-spk-meta { font-size: 11px; color: var(--text-3); margin-top: 1px; }

/* Transcript nodes */
.vd-node {
  margin-bottom: 12px; padding: 14px 18px;
  border-left: 4px solid; border-radius: var(--r-lg);
  background: var(--bg-card); border-top: 1px solid var(--border);
  border-right: 1px solid var(--border); border-bottom: 1px solid var(--border);
  transition: .15s ease;
}
.vd-node:hover { background: rgba(30,41,59,.85); border-color: var(--border-md); }
.vd-node-head { display: flex; align-items: center; gap: 8px; margin-bottom: 6px; }
.vd-dot { width: 8px; height: 8px; border-radius: 50%; display: inline-block; }
.vd-node-head strong { font-size: 13px; font-weight: 700; }
.vd-ts { color: var(--text-3); font-size: 11px; font-family: 'Fira Code',monospace; }
.vd-node-text { font-size: 15px; line-height: 1.65; color: #E2E8F0; word-break: break-word; }
.vd-node-text.vd-rtl { direction: rtl; text-align: right; font-family: 'Noto Nastaliq Urdu',serif; font-size: 17px; line-height: 2.1; color: var(--text-1); }

/* Empty states */
.vd-empty { display: flex; flex-direction: column; align-items: center; justify-content: center; padding: 60px 20px; text-align: center; color: var(--text-3); gap: 8px; min-height: 360px; }
.vd-empty p { font-size: 15px; font-weight: 600; color: var(--text-2); margin: 0; }
.vd-empty span { font-size: 12px; color: var(--text-3); }
.vd-empty-sm { display: flex; flex-direction: column; align-items: center; padding: 24px 12px; text-align: center; color: var(--text-3); gap: 6px; }
.vd-empty-sm p { font-size: 13px; font-weight: 600; color: var(--text-2); margin: 0; }
.vd-empty-sm span { font-size: 11px; color: var(--text-3); }

/* Stats footer */
.vd-stats {
  margin-top: 16px; padding-top: 12px; border-top: 1px solid var(--border);
  font-size: 11px; color: var(--text-3); display: flex; justify-content: space-between;
  font-family: 'Fira Code',monospace;
}

/* Export grid */
.vd-export-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin-top: 10px; }
.vd-export-btn {
  display: flex; align-items: center; gap: 8px;
  padding: 10px 14px; border-radius: var(--r-md);
  background: rgba(255,255,255,.04); border: 1px solid var(--border);
  color: var(--text-1); font-size: 13px; font-weight: 600;
  text-decoration: none; transition: .15s ease; cursor: pointer;
}
.vd-export-btn:hover { background: rgba(255,255,255,.09); border-color: var(--border-md); color: #fff; }
.vd-export-btn svg { color: var(--accent); flex-shrink: 0; }

/* Primary action button */
.vd-btn-primary {
  background: var(--accent-grad) !important; color: #fff !important;
  font-weight: 700 !important; font-size: 14px !important;
  border: none !important; border-radius: var(--r-lg) !important;
  padding: 13px 20px !important; box-shadow: 0 0 20px var(--glow) !important;
  width: 100% !important; cursor: pointer !important; transition: .2s ease !important;
}
.vd-btn-primary:hover { transform: translateY(-2px) !important; box-shadow: 0 0 28px rgba(99,102,241,.5) !important; }

/* Transcript viewport */
.vd-transcript-vp {
  background: var(--bg-base); border: 1px solid var(--border); border-radius: var(--r-xl);
  padding: 20px; min-height: 420px; max-height: 560px; overflow-y: auto;
}
"""

with gr.Blocks(title="VoiceDiary — Bilingual Lecture & Diarization Engine",
               css=CSS, theme=gr.themes.Default(primary_hue="indigo", neutral_hue="slate")) as demo:

    # Hidden state
    transcript_state = gr.State("")

    # ─── HEADER ───
    gr.HTML(f"""
    <header class="vd-header">
      <div class="vd-header-left">
        <div class="vd-logo"><svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2"><path d="M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z"/><path d="M19 10v2a7 7 0 0 1-14 0v-2"/><line x1="12" y1="19" x2="12" y2="23"/><line x1="8" y1="23" x2="16" y2="23"/></svg></div>
        <div><h1 class="vd-title">VoiceDiary</h1><div class="vd-subtitle">AI Bilingual Lecture & Diarization Engine</div></div>
      </div>
      <div class="vd-pill"><span class="vd-pill-dot"></span>{gpu_name} · Tensor Cores {compute_dtype.upper()}</div>
    </header>""")

    with gr.Row():
        # ─── LEFT SIDEBAR ───
        with gr.Column(scale=3):
            with gr.Group():
                gr.HTML("<div class='vd-section-hdr'><span>SPEAKERS & PROFILES</span><span class='vd-badge'>LIVE</span></div>")
                sidebar_out = gr.HTML(value="<div class='vd-empty-sm'><svg width='28' height='28' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.3'><path d='M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z'/><path d='M19 10v2a7 7 0 0 1-14 0v-2'/></svg><p>No speakers detected</p><span>Start recording to identify speakers</span></div>")

            with gr.Group():
                gr.HTML("<div class='vd-section-hdr'><span>AI ENGINE & MODEL HUB</span></div>")
                model_dd = gr.Dropdown(
                    choices=list(MODEL_MAP.keys()),
                    value='Large-v3-Turbo (809M)',
                    label='Active Whisper Model')
                lang_dd = gr.Dropdown(
                    choices=list(LANG_MAP.keys()),
                    value='Bilingual (Urdu + English)',
                    label='Language Output Mode')
                thresh_sl = gr.Slider(20,70,32,step=1,label='Diarization Sensitivity (Cosine %)')
                vad_sl = gr.Slider(150,600,280,step=10,label='VAD Silence Gap (ms)')
                gemini_key = gr.Textbox(placeholder='Paste Gemini API Key…',type='password',label='Gemini AI Key (BYOK)')

        # ─── RIGHT MAIN ───
        with gr.Column(scale=7):
            with gr.Group():
                with gr.Tabs():
                    with gr.TabItem("🎙️ Live Classroom Lecture"):
                        audio_mic = gr.Audio(sources=["microphone"], type="filepath", label="Record Classroom Speech")
                    with gr.TabItem("📁 Upload Audio File"):
                        audio_file = gr.Audio(sources=["upload"], type="filepath", label="Upload Lecture (.wav .mp3 .m4a .flac)")
                transcribe_btn = gr.Button("Transcribe & Diarize Lecture (GPU)", elem_classes=["vd-btn-primary"])

            with gr.Group():
                gr.HTML("<div class='vd-section-hdr'><span>CLASSROOM LECTURE TRANSCRIPT</span></div>")
                transcript_out = gr.HTML(
                    value="<div class='vd-empty'><svg width='40' height='40' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.4'><path d='M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z'/></svg><p>Lecture transcript will stream here</p><span>Record audio or upload a file to begin</span></div>",
                    elem_classes=["vd-transcript-vp"])

            with gr.Group():
                gr.HTML("<div class='vd-section-hdr'><span>EXPORT NOTES</span></div>")
                export_html_out = gr.HTML(value="<div style='color:var(--text-3);font-size:12px;padding:8px 0;'>Export options will appear after transcription.</div>")
                # Hidden file outputs for Gradio file serving
                with gr.Row(visible=False):
                    f_md = gr.File(); f_txt = gr.File(); f_srt = gr.File()

            with gr.Group():
                gr.HTML("<div class='vd-section-hdr'><span>AI STUDY SUMMARY & FLASHCARDS</span></div>")
                ai_btn = gr.Button("✨ Generate AI Summary (Gemini 2.5 Flash)", elem_classes=["vd-btn-primary"])
                ai_out = gr.Markdown(value="*Click above after transcription to generate structured study notes.*")

    # ─── EVENT WIRING ───
    all_inputs = [audio_mic, audio_file, model_dd, lang_dd, thresh_sl, vad_sl]
    all_outputs = [transcript_out, sidebar_out, export_html_out, transcript_state, f_md, f_txt, f_srt]

    def run_from_mic(mic, _file, mod, lang, th, vad):
        return transcribe(mic, mod, lang, th, vad)
    def run_from_file(_mic, fpath, mod, lang, th, vad):
        return transcribe(fpath, mod, lang, th, vad)
    def run_from_btn(mic, fpath, mod, lang, th, vad):
        return transcribe(mic if mic else fpath, mod, lang, th, vad)

    # Auto-transcribe when mic recording stops
    audio_mic.stop_recording(fn=run_from_mic, inputs=all_inputs, outputs=all_outputs)
    # Button for upload tab or manual re-run
    transcribe_btn.click(fn=run_from_btn, inputs=all_inputs, outputs=all_outputs)
    # Gemini
    ai_btn.click(fn=gemini_summary, inputs=[transcript_state, gemini_key], outputs=[ai_out])

demo.queue(max_size=20).launch(share=True, inline=True, debug=False, show_error=True)
